# 05 — Mark 4B — ROI Probability & Threshold Diagnostics

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **05 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_4b_roi_probability_diagnostics.ipynb`

## Objective

With the Mark 4 model frozen, does simply re-thresholding the ROI tumour probabilities recover recall? Sweep the global threshold, per-patient frontiers, calibration and focus volumes (V104, V116).

## Inputs (read-only)

- `mark_4_gate_result.json` + `validation_roi_manifest.csv` from Mark 4
- Frozen Mark 4 best checkpoint (REUSE) or retrained (REBUILD)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_4b_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_4b_gate_result.json` |
| `cache_coverage.csv` |
| `probability_slice_statistics.csv` |
| `threshold_results.csv` |
| `threshold_patient_metrics.csv` |
| `bootstrap_confidence_intervals.csv` |
| `expected_vs_actual.csv` |
| `calibration_frontier_dashboard.png` |
| `patient_threshold_heatmap.png` |
| `focus_probability_histograms.png` |

**Visualizations produced by this notebook:** `calibration_frontier_dashboard.png`, `patient_threshold_heatmap.png`, `focus_probability_histograms.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_4_gate_result.json, Frozen Mark 4 best checkpoint (REUSE) or retrained (REBUILD)"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_4b_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Threshold alone cannot fix recall.** Lowering the threshold trades empty-slice false positives for modest Q1 gains; no global threshold satisfies the continuation targets. The lesion-size Q1 population is systematically under-confident.

## Gate

`mark_4b_gate_result.json` — threshold recalibration gate

## Run notes

Re-predicts from the archived Mark 4 best checkpoint into a frozen probability cache, then sweeps thresholds 0.05–0.70. No retraining.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_4b"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 5 — Mark 4B: ROI Probability Calibration and Failure Diagnostics

**Original:** `mark 1/mark_4b_roi_probability_diagnostics.ipynb`

## Question

Mark 4 missed the positive predicted-empty gate by ~2 percentage points. Does the epoch-5 model
already contain **usable sub-0.50 tumour probabilities** (fixable by thresholding) or is a training
change required?

## Key finding (reproduced)

- **No threshold passed the temporary continuation gate** — positive predicted-empty stays between
  36.3% and 37.0% across the whole grid.
- Selected global threshold 0.60: mean patient Dice 0.3646, V104 0.0646, V116 0.0104,
  Q1 42.59%, positive empty 37.04% (fail), empty FP 3.40% → **5/6 targets**.
- Threshold-only tuning cannot fix the recall failure → Mark 4C changes one causal factor at a time.

## Contract

- One global threshold for every validation patient (no per-patient/per-size thresholds).
- Threshold grid: 0.05, 0.10, 0.15, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70.
- Temporary targets identical to Mark 4 continuation targets. Test split locked.

### 5.1 Verify the Mark 4 evidence and load the frozen checkpoint

In [2]:
import shutil
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

mark4_gate = require_upstream_gate("mark_4")
assert mark4_gate["status"] == "mark_4_smoke_fail"
assert mark4_gate["best_epoch"] == 5
assert mark4_gate["test_images_accessed"] is False

checkpoint = torch.load(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
                        map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 5
assert checkpoint["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
assert checkpoint["source_checkpoint_sha256"] == EXPECTED_SOURCE_CHECKPOINT_SHA256
model = MobileNetV2UNet(in_channels=1, out_channels=1, pretrained=False)
model.load_state_dict(checkpoint["model_state"], strict=True)
model.to(DEVICE).eval()
with torch.inference_mode():
    probe = torch.sigmoid(model(torch.zeros(1, 1, 256, 256, device=DEVICE)))
assert probe.shape == (1, 1, 256, 256) and torch.isfinite(probe).all()
print("PASS: frozen Mark 4 best checkpoint loaded.")

PASS: frozen Mark 4 best checkpoint loaded.


### 5.2 Build the frozen validation ROI loader and cache probabilities

In [3]:
class ValidationROIDataset(Dataset):
    def __init__(self, rows, rois):
        self.rows = rows.reset_index(drop=True)
        self.rois = rois.set_index("volume_id")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        roi = self.rois.loc[int(row.volume_id)]
        y0, y1, x0, x1 = [int(roi[key]) for key in ("y0", "y1", "x0", "x1")]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        image_roi = resize_float(image[y0:y1, x0:x1])
        return {"image": torch.from_numpy(image_roi[None]).float(),
                "sample_id": str(row.sample_id), "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index),
                "box": torch.tensor([y0, y1, x0, x1], dtype=torch.int32)}


validation_rois = pd.read_csv(load_shared("mark_4", "validation_roi_manifest.csv"))
dataset = ValidationROIDataset(validation_manifest, validation_rois)
loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0,
                    pin_memory=torch.cuda.is_available())

ORIG_CACHE = MARK1_DIR / "mark_4b_outputs" / "probability_cache"
CACHE_DIR = OUT_CACHE["mark_4b"]
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def save_volume(volume_id, bucket):
    order = np.argsort(bucket["slice_index"])
    np.savez_compressed(CACHE_DIR / f"volume_{volume_id}.npz",
                        **{key: np.asarray(value)[order] for key, value in bucket.items()})


def build_cache():
    current_volume, bucket, processed = None, None, []
    with torch.inference_mode():
        for batch in loader:
            probabilities = torch.sigmoid(
                model(batch["image"].to(DEVICE, non_blocking=True))).cpu().numpy()[:, 0]
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                if current_volume is None or volume_id != current_volume:
                    if current_volume is not None:
                        save_volume(current_volume, bucket)
                        processed.append(current_volume)
                    current_volume = volume_id
                    bucket = {"sample_id": [], "slice_index": [],
                              "tumor_probability": [], "tumor_truth": []}
                probability_full = probability_to_full(probabilities[index],
                                                       batch["box"][index].numpy())
                row = validation_manifest.loc[
                    validation_manifest["sample_id"].eq(sample_id)].iloc[0]
                with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
                    truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
                bucket["sample_id"].append(str(sample_id))
                bucket["slice_index"].append(int(batch["slice_index"][index]))
                bucket["tumor_probability"].append(probability_full.astype(np.float16))
                bucket["tumor_truth"].append(truth.astype(np.uint8))
    if current_volume is not None:
        save_volume(current_volume, bucket)
        processed.append(current_volume)
    return processed


existing = sorted(CACHE_DIR.glob("volume_*.npz"))
if REUSE_CACHES and len(existing) == 13:
    print(f"REUSE: {len(existing)} patient caches already present in {CACHE_DIR}.")
elif REUSE_CACHES and len(list(ORIG_CACHE.glob("volume_*.npz"))) == 13:
    for p in ORIG_CACHE.glob("volume_*.npz"):
        shutil.copy2(p, CACHE_DIR / p.name)
    print("REUSE: copied 13 frozen caches from mark_4b_outputs/probability_cache.")
else:
    processed = build_cache()
    print(f"REBUILD: cached {len(processed)} volumes.")

m4b_cache, coverage, stats = {}, [], []
for path in sorted(CACHE_DIR.glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        item = {key: payload[key] for key in payload.files}
    expected = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)].sort_values("slice_index")
    assert item["sample_id"].astype(str).tolist() == expected["sample_id"].astype(str).tolist()
    assert np.isfinite(item["tumor_probability"]).all()
    m4b_cache[volume_id] = item
    coverage.append({"volume_id": volume_id, "slices": len(item["sample_id"]),
                     "positive_slices": int(item["tumor_truth"].any(axis=(1, 2)).sum()),
                     "cache_mb": path.stat().st_size / (1024 ** 2)})
    for index, sample_id in enumerate(item["sample_id"]):
        truth = item["tumor_truth"][index].astype(bool)
        probability = item["tumor_probability"][index].astype(np.float32)
        inside = probability[truth]
        stats.append({"sample_id": str(sample_id), "volume_id": volume_id,
                      "slice_index": int(item["slice_index"][index]),
                      "true_pixels": int(truth.sum()),
                      "max_probability": float(probability.max()),
                      "mean_probability_inside_truth": float(inside.mean()) if inside.size else np.nan,
                      "max_probability_inside_truth": float(inside.max()) if inside.size else np.nan})
coverage_frame = pd.DataFrame(coverage)
m4b_stats = pd.DataFrame(stats)
assert coverage_frame["slices"].sum() == 10_685
coverage_frame.to_csv(OUT["mark_4b"] / "cache_coverage.csv", index=False)
m4b_stats.to_csv(OUT["mark_4b"] / "probability_slice_statistics.csv", index=False)
display(coverage_frame)
print("PASS: 13 complete probability caches validated.")

REUSE: copied 13 frozen caches from mark_4b_outputs/probability_cache.


,volume_id,slices,positive_slices,cache_mb
0,104,781,122,0.182208
1,105,986,0,0.190859
2,106,771,0,0.151744
3,107,771,71,0.173355
4,108,856,202,1.504140
5,109,756,131,0.247720
6,110,816,130,0.285200
7,111,761,92,0.170486
8,112,751,26,0.154068
9,113,836,135,0.286286


PASS: 13 complete probability caches validated.


### 5.3 Evaluate the global threshold grid

In [4]:
THRESHOLD_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.35, 0.40, 0.45,
                           0.50, 0.55, 0.60, 0.70], dtype=np.float32)

positive_size_reference = m4b_stats.loc[m4b_stats["true_pixels"].gt(0),
                                        ["sample_id", "true_pixels"]].copy()
positive_size_reference["size_quartile"] = pd.qcut(
    positive_size_reference["true_pixels"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
size_map = positive_size_reference.set_index("sample_id")["size_quartile"]


def evaluate_threshold(threshold):
    patient_rows, slice_rows = [], []
    total_intersection = total_predicted = total_true = 0
    for volume_id, item in m4b_cache.items():
        truth = item["tumor_truth"].astype(bool)
        prediction = item["tumor_probability"].astype(np.float32) >= threshold
        intersections = (prediction & truth).sum(axis=(1, 2))
        predicted = prediction.sum(axis=(1, 2))
        true = truth.sum(axis=(1, 2))
        total_intersection += int(intersections.sum())
        total_predicted += int(predicted.sum())
        total_true += int(true.sum())
        patient_rows.append({
            "volume_id": volume_id, "true_pixels": int(true.sum()),
            "predicted_pixels": int(predicted.sum()),
            "intersection_pixels": int(intersections.sum()),
            "micro_dice": (2 * intersections.sum() + 1e-6) / (predicted.sum() + true.sum() + 1e-6),
            "positive_predicted_empty_pct": 100 * int(((true > 0) & (predicted == 0)).sum())
                                            / max(int((true > 0).sum()), 1),
            "empty_slice_false_positive_pct": 100 * int(((true == 0) & (predicted > 0)).sum())
                                              / max(int((true == 0).sum()), 1)})
        for index, sample_id in enumerate(item["sample_id"]):
            slice_rows.append({"sample_id": str(sample_id), "volume_id": volume_id,
                               "true_pixels": int(true[index]),
                               "predicted_pixels": int(predicted[index]),
                               "intersection_pixels": int(intersections[index])})
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = positive_slices["sample_id"].map(size_map)
    q1 = positive_slices.loc[positive_slices["size_quartile"].eq("Q1")]
    return {
        "threshold": float(threshold),
        "global_dice": (2 * total_intersection + 1e-6) / (total_predicted + total_true + 1e-6),
        "pixel_precision": total_intersection / max(total_predicted, 1),
        "pixel_recall": total_intersection / max(total_true, 1),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": 100 * float((q1["intersection_pixels"] > 0).mean()),
        "positive_predicted_empty_pct": 100 * float(
            (positive_slices["predicted_pixels"] == 0).sum() / len(positive_slices)),
        "empty_slice_false_positive_pct": 100 * float(
            ((slices["true_pixels"] == 0) & (slices["predicted_pixels"] > 0)).sum()
            / max((slices["true_pixels"] == 0).sum(), 1)),
        "predicted_tumor_pixels": total_predicted,
    }, patients, slices


result_rows, patient_frames = [], []
for threshold in THRESHOLD_GRID:
    result, patients, slices = evaluate_threshold(float(threshold))
    result_rows.append(result)
    patient_frames.append(patients.assign(threshold=float(threshold)))
m4b_threshold_results = pd.DataFrame(result_rows)
m4b_patient_metrics = pd.concat(patient_frames, ignore_index=True)
m4b_threshold_results.to_csv(OUT["mark_4b"] / "threshold_results.csv", index=False)
m4b_patient_metrics.to_csv(OUT["mark_4b"] / "threshold_patient_metrics.csv", index=False)
display(m4b_threshold_results)

,threshold,global_dice,pixel_precision,pixel_recall,mean_patient_dice,median_patient_dice,worst_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,predicted_tumor_pixels
0,0.05,0.575070,0.658503,0.510401,0.341725,0.338068,0.013755,0.079753,0.013755,42.965779,36.276392,3.536244,511164
1,0.10,0.576417,0.681951,0.499168,0.347554,0.354157,0.013125,0.076838,0.013125,42.965779,36.276392,3.505133,482725
2,0.15,0.576168,0.696987,0.491047,0.351847,0.368941,0.012684,0.074972,0.012684,42.965779,36.372361,3.494763,464627
3,0.20,0.575432,0.708222,0.484575,0.354422,0.376052,0.012420,0.073760,0.012420,42.965779,36.372361,3.474023,451230
4,0.30,0.573053,0.726605,0.473078,0.358289,0.389446,0.011785,0.071542,0.011785,42.965779,36.468330,3.453282,429379
5,0.35,0.571463,0.734589,0.467621,0.360205,0.397665,0.011441,0.070721,0.011441,42.585551,36.564299,3.432542,419813
6,0.40,0.569402,0.741911,0.461982,0.361614,0.404975,0.011226,0.069440,0.011226,42.585551,36.756238,3.422172,410657
7,0.45,0.566999,0.748845,0.456214,0.362856,0.410082,0.010997,0.068297,0.010997,42.585551,36.852207,3.422172,401775
8,0.50,0.564202,0.755802,0.450100,0.363902,0.415101,0.010780,0.067186,0.010780,42.585551,36.852207,3.422172,392742
9,0.55,0.561277,0.762425,0.444109,0.364448,0.419625,0.010616,0.065954,0.010616,42.585551,36.852207,3.411801,384148


### 5.4 Apply continuation and final-target gates

In [5]:
def pass_columns(frame, targets, prefix):
    pass_frame = pd.DataFrame(index=frame.index)
    for key, target in targets.items():
        if key in ("positive_predicted_empty_pct", "empty_slice_false_positive_pct"):
            pass_frame[f"{prefix}_{key}"] = frame[key] <= target
        else:
            pass_frame[f"{prefix}_{key}"] = frame[key] >= target
    return pass_frame


cont_flags = pass_columns(m4b_threshold_results, CONTINUATION_TARGETS, "continue")
final_flags = pass_columns(m4b_threshold_results, FINAL_TARGETS, "final")
m4b_threshold_results["continuation_targets_passed"] = cont_flags.sum(axis=1)
m4b_threshold_results["all_continuation_targets_passed"] = cont_flags.all(axis=1)
m4b_threshold_results["final_targets_passed"] = final_flags.sum(axis=1)
m4b_threshold_results["all_final_targets_passed"] = final_flags.all(axis=1)

eligible = m4b_threshold_results.loc[m4b_threshold_results["all_continuation_targets_passed"]]
if not eligible.empty:
    selected = eligible.sort_values(["mean_patient_dice", "empty_slice_false_positive_pct"],
                                    ascending=[False, True]).iloc[0]
else:
    selected = m4b_threshold_results.sort_values(
        ["continuation_targets_passed", "mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, False, True]).iloc[0]
m4b_threshold_results.to_csv(OUT["mark_4b"] / "threshold_results.csv", index=False)
print("Selected diagnostic threshold:", selected.to_dict())

Selected diagnostic threshold: {'threshold': 0.6000000238418579, 'global_dice': 0.5580796081026355, 'pixel_precision': 0.7686127405988534, 'pixel_recall': 0.43808293415943, 'mean_patient_dice': 0.3645500967728056, 'median_patient_dice': 0.42483907621642575, 'worst_patient_dice': 0.010424480722403422, 'volume_104_dice': 0.06455658243962056, 'volume_116_dice': 0.010424480722403422, 'q1_detected_pct': 42.585551330798474, 'positive_predicted_empty_pct': 37.04414587332054, 'empty_slice_false_positive_pct': 3.401431089909779, 'predicted_tumor_pixels': 375885, 'continuation_targets_passed': 5, 'all_continuation_targets_passed': False, 'final_targets_passed': 1, 'all_final_targets_passed': False}


### 5.5 Calibration frontiers, patient heatmap, localization, populations

In [6]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
axes[0, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["mean_patient_dice"], marker="o")
axes[0, 0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 0].set_title("Mean patient Dice")
axes[0, 1].plot(m4b_threshold_results["threshold"], m4b_threshold_results["volume_104_dice"], marker="o", label="V104")
axes[0, 1].plot(m4b_threshold_results["threshold"], m4b_threshold_results["volume_116_dice"], marker="s", label="V116")
axes[0, 1].set_title("Focus-patient Dice"); axes[0, 1].legend()
axes[0, 2].plot(m4b_threshold_results["threshold"], m4b_threshold_results["q1_detected_pct"], marker="o")
axes[0, 2].axhline(CONTINUATION_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[0, 2].set_title("Q1 detection (%)")
axes[1, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["positive_predicted_empty_pct"], marker="o", label="Positive empty")
axes[1, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["empty_slice_false_positive_pct"], marker="s", label="Empty FP")
axes[1, 0].legend(); axes[1, 0].set_title("Slice error rates (%)")
axes[1, 1].plot(m4b_threshold_results["pixel_recall"], m4b_threshold_results["pixel_precision"], marker="o")
for row in m4b_threshold_results.itertuples():
    axes[1, 1].annotate(f"{row.threshold:.2f}", (row.pixel_recall, row.pixel_precision), fontsize=7)
axes[1, 1].set_xlabel("Recall"); axes[1, 1].set_ylabel("Precision")
axes[1, 1].set_title("Pixel precision–recall")
axes[1, 2].scatter(m4b_threshold_results["empty_slice_false_positive_pct"],
                   m4b_threshold_results["mean_patient_dice"],
                   c=m4b_threshold_results["threshold"], cmap="viridis", s=65)
axes[1, 2].set_xlabel("Empty-slice FP (%)"); axes[1, 2].set_ylabel("Mean patient Dice")
axes[1, 2].set_title("Validation frontier")
for axis in axes.flat:
    axis.set_xlabel(axis.get_xlabel() or "Global threshold")
    axis.grid(alpha=0.25)
figure.suptitle("Mark 4B global threshold diagnostics", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4b"] / "calibration_frontier_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

heatmap = m4b_patient_metrics.pivot(index="volume_id", columns="threshold", values="micro_dice")
figure, axis = plt.subplots(figsize=(15, 6))
image = axis.imshow(heatmap, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axis.set_xticks(range(len(heatmap.columns)), [f"{v:.2f}" for v in heatmap.columns])
axis.set_yticks(range(len(heatmap.index)), heatmap.index)
axis.set_xlabel("Global threshold"); axis.set_ylabel("Volume")
axis.set_title("Patient Dice across thresholds")
figure.colorbar(image, ax=axis, label="Micro-Dice")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4b"] / "patient_threshold_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()


def load_full_image(sample_id):
    row = validation_manifest.loc[validation_manifest["sample_id"].eq(sample_id)].iloc[0]
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        return np.asarray(handle.convert("L"), dtype=np.float32) / 255.0


def localization_panel(volume_id):
    item = m4b_cache[volume_id]
    sizes = item["tumor_truth"].sum(axis=(1, 2))
    positive_indices = np.flatnonzero(sizes > 0)
    chosen = list(dict.fromkeys([
        int(positive_indices[np.argmax(sizes[positive_indices])]),
        int(positive_indices[len(positive_indices) // 2]),
        int(positive_indices[np.argmin(sizes[positive_indices])]),
    ]))
    figure, axes = plt.subplots(len(chosen), 5, figsize=(18, 3.7 * len(chosen)))
    if len(chosen) == 1:
        axes = axes[None, :]
    threshold = float(selected["threshold"])
    for row_axes, index in zip(axes, chosen):
        sample_id = str(item["sample_id"][index])
        image = load_full_image(sample_id)
        truth = item["tumor_truth"][index].astype(bool)
        probability = item["tumor_probability"][index].astype(np.float32)
        prediction = probability >= threshold
        error = np.zeros((*truth.shape, 3), dtype=np.float32)
        error[truth & ~prediction, 0] = 1
        error[prediction & ~truth, 2] = 1
        panels = [(image, "CT", "gray"), (truth, "Truth", "gray"),
                  (probability, "Probability", "magma"),
                  (prediction, f"Prediction t={threshold:.2f}", "gray"),
                  (error, "FN red / FP blue", None)]
        for axis, (panel, title, cmap) in zip(row_axes, panels):
            axis.imshow(panel, cmap=cmap, vmin=0 if panel.ndim == 2 else None,
                        vmax=1 if panel.ndim == 2 else None)
            axis.set_title(title); axis.axis("off")
        row_axes[0].set_ylabel(sample_id, fontsize=8)
    figure.suptitle(f"Volume {volume_id} localization", fontsize=17, weight="bold")
    figure.tight_layout()
    figure.savefig(OUT_FIGS["mark_4b"] / f"localization_volume_{volume_id}.png",
                   dpi=170, bbox_inches="tight")
    plt.show()


localization_panel(104)
localization_panel(116)

figure, axes = plt.subplots(2, 1, figsize=(12, 8))
bins = np.linspace(0, 1, 51)
rng = np.random.default_rng(SEED)
for axis, volume_id in zip(axes, [104, 116]):
    item = m4b_cache[volume_id]
    probability = item["tumor_probability"].astype(np.float32)
    truth = item["tumor_truth"].astype(bool)
    true_values = probability[truth]
    background = probability[~truth]
    if background.size > 500_000:
        background = rng.choice(background, 500_000, replace=False)
    axis.hist(true_values, bins=bins, density=True, histtype="step", linewidth=2,
              label=f"True tumor n={len(true_values):,}")
    axis.hist(background, bins=bins, density=True, histtype="step", linewidth=1.5,
              label=f"Background sample n={len(background):,}")
    axis.set_yscale("log"); axis.set_xlim(0, 1)
    axis.set_title(f"Volume {volume_id}"); axis.legend()
figure.suptitle("Focus-patient probability populations", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4b"] / "focus_probability_histograms.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_18628\2173164949.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_18628\2173164949.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_18628\2173164949.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_18628\2173164949.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.6 Bootstrap uncertainty + write the Mark 4B gate

In [7]:
selected_patients = m4b_patient_metrics.loc[
    m4b_patient_metrics["threshold"].eq(float(selected["threshold"]))
    & m4b_patient_metrics["true_pixels"].gt(0)].copy()
values = selected_patients["micro_dice"].to_numpy()
rng = np.random.default_rng(SEED)
bootstrap_means = np.array([rng.choice(values, size=len(values), replace=True).mean()
                            for _ in range(1_000)])
m4b_bootstrap = pd.DataFrame([{
    "threshold": float(selected["threshold"]), "patients": len(values),
    "iterations": 1_000, "mean_dice_p2_5": np.percentile(bootstrap_means, 2.5),
    "mean_dice_p50": np.percentile(bootstrap_means, 50),
    "mean_dice_p97_5": np.percentile(bootstrap_means, 97.5),
    "original_median": np.median(values), "original_q25": np.percentile(values, 25),
    "original_q75": np.percentile(values, 75)}])
m4b_bootstrap.to_csv(OUT["mark_4b"] / "bootstrap_confidence_intervals.csv", index=False)
display(m4b_bootstrap)

continuation_passed = bool(selected["all_continuation_targets_passed"])
final_passed = bool(selected["all_final_targets_passed"])
baseline_050 = m4b_threshold_results.loc[np.isclose(m4b_threshold_results["threshold"], 0.50)].iloc[0]

if continuation_passed:
    decision, next_notebook = "FREEZE_THRESHOLD_AND_PROCEED_TO_BOUNDED_EPOCH_10_CONTINUATION", "mark_5_two_stage_bounded_continuation"
elif (selected["positive_predicted_empty_pct"] <= CONTINUATION_TARGETS["positive_predicted_empty_pct"]
      and selected["empty_slice_false_positive_pct"] > CONTINUATION_TARGETS["empty_slice_false_positive_pct"]):
    decision, next_notebook = "REVISE_SAMPLING_OR_STABLE_RECALL_OBJECTIVE", "mark_4c_sampling_loss_ablation"
else:
    decision, next_notebook = "PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT_OR_OBJECTIVE", "mark_4c_two_channel_or_recall_ablation"

m4b_gate = {
    "status": "mark_4b_diagnostic_complete",
    "selected_global_threshold": float(selected["threshold"]),
    "continuation_gate_passed": continuation_passed,
    "final_validation_gate_passed": final_passed,
    "selected_metrics": {key: float(selected[key]) for key in CONTINUATION_TARGETS},
    "targets_passed": int(selected["continuation_targets_passed"]),
    "threshold_0_50_metrics": {key: float(baseline_050[key]) for key in CONTINUATION_TARGETS},
    "bootstrap": m4b_bootstrap.iloc[0].to_dict(),
    "decision": decision, "next_notebook": next_notebook,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "checkpoint_sha256": sha256_file(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth"),
    "test_images_accessed": False,
}
(OUT["mark_4b"] / "mark_4b_gate_result.json").write_text(json.dumps(m4b_gate, indent=2))
expected_actual = pd.DataFrame([
    {"metric": key, "actual": selected[key],
     "continuation_target": CONTINUATION_TARGETS[key], "final_target": FINAL_TARGETS[key]}
    for key in CONTINUATION_TARGETS])
expected_actual.to_csv(OUT["mark_4b"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([m4b_gate]).T.rename(columns={0: "value"}))
display(expected_actual)
print(decision)

# ---- Reproduction check against the original gate ----
orig_m4b = json.loads((MARK1_DIR / "mark_4b_outputs" / "mark_4b_gate_result.json").read_text())
diffs = {k: abs(float(m4b_gate["selected_metrics"][k]) - float(orig_m4b["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4B reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4B gate drifted from the original!"
assert abs(float(m4b_gate["selected_global_threshold"]) - float(orig_m4b["selected_global_threshold"])) < 1e-5
print("PASS: Mark 4B gate matches the original mark_4b_gate_result.json.")

,threshold,patients,iterations,mean_dice_p2_5,mean_dice_p50,mean_dice_p97_5,original_median,original_q25,original_q75
0,0.6,9,1000,0.189341,0.368614,0.531381,0.424839,0.150339,0.52821


,value
status,mark_4b_diagnostic_complete
selected_global_threshold,0.6
continuation_gate_passed,False
final_validation_gate_passed,False
selected_metrics,"{'mean_patient_dice': 0.3645500967728056, 'vol..."
targets_passed,5
threshold_0_50_metrics,"{'mean_patient_dice': 0.363902230361049, 'volu..."
bootstrap,"{'threshold': 0.6000000238418579, 'patients': ..."
decision,PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT...
next_notebook,mark_4c_two_channel_or_recall_ablation


,metric,actual,continuation_target,final_target
0,mean_patient_dice,0.364550,0.3329,0.406915
1,volume_104_dice,0.064557,0.0500,0.500000
2,volume_116_dice,0.010424,0.0100,0.050000
3,q1_detected_pct,42.585551,35.0000,45.000000
4,positive_predicted_empty_pct,37.044146,35.0000,20.000000
5,empty_slice_false_positive_pct,3.401431,20.0000,15.000000


PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT_OR_OBJECTIVE
Mark 4B reproduction check: {'mean_patient_dice': 0.0, 'volume_104_dice': 0.0, 'volume_116_dice': 0.0, 'q1_detected_pct': 0.0, 'positive_predicted_empty_pct': 0.0, 'empty_slice_false_positive_pct': 0.0}
PASS: Mark 4B gate matches the original mark_4b_gate_result.json.


In [8]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_4b")


PASS: mark_4b_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\05_mark_4b\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_18628\3925602944.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ---------------------------------------------------------------------------
# Publish key mark_4b artifacts to the shared/legacy folder other code reads.
#
# External consumers: `mark 1/MARK_1_PROGRESS_TRACKER.md` and downstream tracking documents.
# These code files hardcode artifact paths under `mark 1/mark_4b_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_4b_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_4b"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_4b artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_4b artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 7 mark_4b artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\bootstrap_confidence_intervals.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\cache_coverage.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\expected_vs_actual.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\mark_4b_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\probability_slice_statistics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\threshold_patient_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4b_outputs\threshold_results.csv
